# 1. lm studio langchain 및 retriever 활용 generator 생성 테스트

In [ ]:
import os
from typing import List
from openai import OpenAI

from retriever_factory import get_dense_retriever  # 네가 만든 함수

# =========================
# 프롬프트 템플릿
# =========================
RAG_PROMPT_TEMPLATE = (
"""
절대 한국어로만 답변하십시오. 영어를 사용하지 마십시오.  
You must always respond in Korean. Do not use English.  

당신은 대한민국 현행 법령을 기반으로 답변하는 RAG 챗봇입니다.  
아래 지침을 따르고, 명확하고 간결한 답변을 제공하세요.

- 질문이 단순하면 한 문장으로 답변하세요.  
- 질문이 복잡하면 2~3문장 이내로 답변하세요.  
- 법 조항을 직접 나열하지 말고, 질문과 관련된 핵심 내용을 요약하여 자연스럽게 설명하세요.  
- 각 문장에 해당하는 법적 근거 조항을 괄호 안에 정확히 명시하세요. (예: 도로교통법 제12조 제1항)  
- 질문을 반복하지 말고, 불필요한 형식적인 표현을 포함하지 마세요.  
- **정확도, 확률, 신뢰도 같은 수치를 답변에 절대 포함하지 마세요.**  
- Context 내에서만 답변을 생성하고, 모르는 내용은 `"제가 제공할 수 있는 정보가 없습니다."`라고만 답변하세요.  

{question}에 대한 답변을 다음 Context를 참고하여 작성하세요.

Context:  
{context}  

답변:
"""
)

SYS_PROMPT_TEMPLATE = (
"""
당신은 대한민국 현행 법령을 기반으로 작동하는 RAG 챗봇입니다.  
절대 한국어로만 답변하십시오. 영어를 사용하지 마십시오.  

답변을 생성할 때, 다음 원칙을 따르세요.

1. **간결한 답변 제공**  
   - 단순한 질문 → 한 문장  
   - 복잡한 질문 → 2~3문장  
   - 불필요한 설명, 배경 정보, 반복 표현 금지  

2. **법 조항 요약 방식**  
   - 법 조항을 직접 나열하지 않고 질문과 관련된 핵심 내용을 요약  
   - 각 문장에 해당하는 법적 근거 조항을 괄호 안에 명확히 표기 (예: 도로교통법 제17조 제2항)

3. **불필요한 정보 제한**  
   - "정확도", "신뢰도", "확률" 등의 수치는 절대 포함 금지  
   - "또한", "추가로", "이 외에도"와 같은 확장 표현 사용 금지  

4. **추론 및 임의 해석 금지**  
   - 제공된 Context 범위 내에서만 답변 작성  
   - 필요한 정보가 없으면 `"제가 제공할 수 있는 정보가 없습니다."`라고만 답변  
"""
)

# =========================
# LM Studio용 클라이언트
# =========================
def get_lmstudio_client(base_url: str = "http://localhost:1234/v1") -> OpenAI:
    """
    LM Studio는 OpenAI 호환 API를 쓰니까,
    base_url + api_key="lm-studio"로 호출해주면 됨.
    """
    client = OpenAI(
        base_url=base_url,
        api_key="lm-studio",
    )
    return client

# =========================
# LLM 호출 함수
# =========================
def call_llm_with_context(
    question: str,
    context: str,
    base_url: str = "http://localhost:1234/v1",
    model_name: str = "teddylee777/EEVE-Korean-Instruct-10.8B-v1.0-gguf",
) -> str:
    client = get_lmstudio_client(base_url)

    user_content = RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
    )

    messages = [
        {"role": "system", "content": SYS_PROMPT_TEMPLATE},
        {"role": "user", "content": user_content},
    ]

    completion = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.1,
    )
    return completion.choices[0].message.content

# =========================
# 전체 파이프라인
# =========================
def run_pipeline(
    question: str,
    llm_base_url: str = "http://localhost:1234/v1",
    top_k: int = 10,
    device: str = "cpu",
):
    """
    - retriever_factory.get_dense_retriever() 사용
    - re-ranker 없이 retriever 결과를 그대로 context로 사용
    - 반환: (답변 텍스트, retrieved_docs, used_docs)
    """
    if not question.strip():
        return "질문을 입력해주세요.", [], []

    # 1) Dense retriever 로드 (네가 만든 팩토리 함수 사용)
    retriever = get_dense_retriever(device=device, top_k=top_k)

    # 2) 검색
    retrieved_docs = retriever.invoke(question)
    if not retrieved_docs:
        return "제가 제공할 수 있는 정보가 없습니다.", [], []

    # 3) Context 생성 (필요하면 메타데이터도 같이 붙여도 됨)
    #    일단은 내용만 합치는 단순 버전
    context_chunks: List[str] = []
    for d in retrieved_docs:
        context_chunks.append(d.page_content)
    context = "\n\n".join(context_chunks)

    # 4) LLM 호출
    response = call_llm_with_context(
        question=question,
        context=context,
        base_url=llm_base_url,
    )

    # 인터페이스 맞추려고 세 번째 값도 retrieved_docs 그대로
    return response, retrieved_docs, retrieved_docs

# =========================
# 간단 실행 예시
# =========================
if __name__ == "__main__":
    q = "혈중 알코올 농도가 몇 이어야 음주운전인지 각 처벌 규정을 알려주세요."
    answer, docs, used = run_pipeline(
        question=q,
        llm_base_url="http://localhost:1234/v1",
        top_k=10,
        device="mps",   # M1이면 "mps"로 retriever 올려도 됨
    )

    print("=== 답변 ===")
    print(answer)
    print("\n=== 사용된 문서 수 ===", len(docs))

# 2. 함수 모듈화 및 테스트

In [ ]:
from pipeline import run_pipeline

answer, docs, used = run_pipeline("혈중 알코올 농도가 몇 이어야 음주운전인지 각 처벌 규정을 알려주세요.")
print(answer)